In [11]:
import os
import re
import json
import uuid
import asyncio
import aiohttp
import ssl
import certifi
import pandas as pd
from itertools import combinations
from datetime import datetime
from tqdm.asyncio import tqdm
import threading


In [12]:

# === Setup ===
# OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_KEY = "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"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Validate API key is present
if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY environment variable is not set. "
        "Please set it using: os.environ['OPENROUTER_API_KEY'] = 'your-key-here'"
    )

ALL_MODELS = ["x-ai/grok-4-fast", "google/gemini-2.5-flash", "deepseek/deepseek-chat-v3.1", 
              "openai/gpt-5-nano", "anthropic/claude-sonnet-4.5", "openai/gpt-5"]
MODELS = ["x-ai/grok-4-fast", "google/gemini-2.5-flash", "deepseek/deepseek-chat-v3.1", "openai/gpt-5-nano"]
OUTPUT_DIR = "strategy_two_results"
MAX_CONCURRENT = 2
TEMPERATURE = [0.1, 0.7]

# Strategy 2: Hospital capacity contexts
CAPACITY_CONTEXTS = [20, 50, 80]  # Percentage of hospital beds currently occupied

# Optional: Add your site URL and name for rankings on openrouter.ai
SITE_URL = ""
SITE_NAME = ""

os.makedirs(OUTPUT_DIR, exist_ok=True)
csv_lock = threading.Lock()


In [13]:

# === Canonical Options ===
options = [
    "Option A. No businesses are closed; 1 in 100 people will be infected during the next month.",
    "Option B. No businesses are closed; 10 in 100 people will be infected during the next month.",
    "Option C. No businesses are closed; 20 in 100 people will be infected during the next month.",
    "Option D. Crowded, high-risk businesses (such as clubs, theaters, and large sports venues) are closed; 1 in 100 people will be infected during the next month.",
    "Option E. Crowded, high-risk businesses (such as clubs, theaters, and large sports venues) are closed; 10 in 100 people will be infected during the next month.",
    "Option F. Crowded, high-risk businesses (such as clubs, theaters, and large sports venues) are closed; 20 in 100 people will be infected during the next month.",
    "Option G. All non-essential businesses (such as clubs, theaters, restaurants, bars, and gyms) are closed; 1 in 100 people will be infected during the next month.",
    "Option H. All non-essential businesses (such as clubs, theaters, restaurants, bars, and gyms) are closed; 10 in 100 people will be infected during the next month.",
    "Option I. All non-essential businesses (such as clubs, theaters, restaurants, bars, and gyms) are closed; 20 in 100 people will be infected during the next month.",
]

# Generate pairs in both orders
base_option_pairs = list(combinations(options, 2))
option_pairs = []
for pair in base_option_pairs:
    option_pairs.append(pair)  # Original order (A, B)
    option_pairs.append((pair[1], pair[0]))  # Reversed order (B, A)

canonical_option_map = {re.search(r"Option\s+([A-I])", o).group(1): o for o in options}


In [14]:

# === Prompt Builder for Strategy 2 ===
def build_prompt(pair, capacity_context):
    """Build a prompt with numerical labels and hospital capacity context."""
    # Remove the original letter labels from the option text
    option_1_text = re.sub(r"Option\s+[A-I]\.\s*", "", pair[0])
    option_2_text = re.sub(r"Option\s+[A-I]\.\s*", "", pair[1])
    
    prompt = (
        "I am a public health official for a county in the United States. "
        "A novel viral pathogen (similar to H1N1 and SARS-CoV-2) is spreading in my jurisdiction. "
        "Among those who get infected, 5% would become so sick that they require hospitalization. "
        f"Our hospital capacity is 15 beds per 100,000 population, and {capacity_context}% of the hospital beds are currently occupied. "
        "We are considering closing non-essential businesses to reduce the spread of the pathogen over the next month. "
        "Our goal is to balance the public health benefits and the economic costs on local businesses and workers. "
        "Given this objective, which of the following two options would you choose?\n\n"
        f"Option 1: {option_1_text}\n"
        f"Option 2: {option_2_text}\n\n"
        "Please reply with only the number (1 or 2) of the option you would choose. Do not include any additional text."
    )
    
    return prompt


In [15]:

# === Async OpenRouter Query ===
async def query_openrouter_async(session, model, prompt, temperature=0.1):
    """Async query to OpenRouter API."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    
    if SITE_URL:
        headers["HTTP-Referer"] = SITE_URL
    if SITE_NAME:
        headers["X-Title"] = SITE_NAME
    
    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": temperature
    }
    
    try:
        async with session.post(
            OPENROUTER_URL,
            headers=headers, 
            json=payload, 
            timeout=aiohttp.ClientTimeout(total=60)
        ) as response:
            response_text = await response.text()
            
            if response.status != 200:
                print(f"[Error] HTTP {response.status}: {response_text[:200]}")
                return ""
            
            try:
                result = json.loads(response_text)
                return result["choices"][0]["message"]["content"].strip()
            except (json.JSONDecodeError, KeyError) as e:
                print(f"[Error] Failed to parse response: {e}")
                print(f"[Debug] Response text: {response_text[:200]}")
                return ""
                
    except (aiohttp.ClientError, asyncio.TimeoutError) as e:
        print(f"[Error] Request failed: {e}")
        return ""


In [16]:

# === Parsing ===
def parse_choice(response_text, pair):
    """Parse the numerical choice (1 or 2) and map back to original letter."""
    if not response_text:
        return None
    
    clean_response = response_text.strip()
    
    if clean_response in ['1', '2']:
        choice_num = int(clean_response)
        selected_option = pair[choice_num - 1]
        return re.search(r"Option\s+([A-I])", selected_option).group(1)
    
    patterns = [
        r'^\s*([12])\s*[.,;:!]?\s*$',
        r'Option\s+([12])\b',
        r'(?:choose|select|prefer|pick)\s+(?:Option\s+)?([12])\b',
        r'(?:My choice is|choice is|choice:)\s+(?:Option\s+)?([12])\b',
        r'\b([12])\b',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, response_text, re.IGNORECASE | re.MULTILINE)
        if match:
            choice_num = int(match.group(1))
            if choice_num in [1, 2]:
                selected_option = pair[choice_num - 1]
                return re.search(r"Option\s+([A-I])", selected_option).group(1)
    
    return None


In [17]:

# === Infection/NPI helpers ===
def extract_infection_rate(option_text):
    match = re.search(r"(\d+)", option_text)
    return int(match.group(1)) if match else None

def categorize_closure(option_text):
    if "No businesses are closed" in option_text:
        return "No closure"
    elif "Crowded, high-risk businesses" in option_text:
        return "Crowded closures"
    elif "All non-essential businesses" in option_text:
        return "Non-essential closures"
    return "Other"

def npi_level(option_text):
    cat = categorize_closure(option_text)
    return {"No closure": 0, "Crowded closures": 1, "Non-essential closures": 2}.get(cat, -1)

# === CSV Logger for Strategy 2 ===
def log_response_to_csv(model, pair, selected_choice, response_text, temperature, capacity_context, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    selected_option = canonical_option_map.get(selected_choice)
    if not selected_option:
        print(f"[Warning] Could not find canonical option for {selected_choice}")
        return

    non_selected_option = next((o for o in pair if o != selected_option), None)
    infection_rate_sel = extract_infection_rate(selected_option)
    npi_sel = npi_level(selected_option)
    infection_rate_non = extract_infection_rate(non_selected_option) if non_selected_option else None
    npi_non = npi_level(non_selected_option) if non_selected_option else None
    option_letters = [re.search(r"Option\s+([A-I])", o).group(1) for o in pair]

    entry_id = (
        f"{model.replace('/', '_').replace(':', '_')}_{option_letters[0]}_{option_letters[1]}_"
        f"cap{capacity_context}_temp{temperature}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid.uuid4().hex[:8]}"
    )

    record = {
        "id": entry_id,
        "timestamp": datetime.utcnow().isoformat(),
        "model": model,
        "temperature": temperature,
        "capacity_context": capacity_context,
        "pair": f"{option_letters[0]}-{option_letters[1]}",
        "option_1": pair[0],
        "option_2": pair[1],
        "selected_choice": selected_choice,
        "response_text": response_text,
        "infection_rate_selected": infection_rate_sel,
        "npi_level_selected": npi_sel,
        "infection_rate_non_selected": infection_rate_non,
        "npi_level_non_selected": npi_non,
    }

    csv_path = os.path.join(output_dir, f"results_{model.replace('/', '_').replace(':', '_')}.csv")
    with csv_lock:
        df = pd.DataFrame([record])
        if not os.path.exists(csv_path):
            df.to_csv(csv_path, index=False)
        else:
            df.to_csv(csv_path, mode="a", index=False, header=False)


In [18]:

# === Async Process Single Pair with Retry Logic ===
async def process_pair_async(session, model, pair, temperature, capacity_context, semaphore, max_retries=3):
    """Process with semaphore to control concurrency and retry on failure."""
    async with semaphore:
        prompt = build_prompt(pair, capacity_context)
        
        for attempt in range(1, max_retries + 1):
            response = await query_openrouter_async(session, model, prompt, temperature)
            
            if not response:
                if attempt < max_retries:
                    wait_time = 2 ** attempt
                    await asyncio.sleep(wait_time)
                    continue
                else:
                    return False, (model, pair, temperature, capacity_context)
            
            choice_letter = parse_choice(response, pair)
            if not choice_letter:
                if attempt < max_retries:
                    await asyncio.sleep(2 ** attempt)
                    continue
                else:
                    print(f"[Warning] Invalid response after {max_retries} attempts for {model} (temp={temperature}, cap={capacity_context}%): {response[:120]}")
                    return False, (model, pair, temperature, capacity_context)

            log_response_to_csv(model, pair, choice_letter, response, temperature, capacity_context, OUTPUT_DIR)
            return True, None
        
        return False, (model, pair, temperature, capacity_context)


In [19]:

# === Async Simulation ===
async def run_simulation_async():
    print(f"Starting simulation with models: {', '.join(MODELS)}")
    print(f"Temperature values: {TEMPERATURE}")
    print(f"Capacity contexts: {CAPACITY_CONTEXTS}%")
    print(f"Max concurrent requests: {MAX_CONCURRENT}")
    print(f"Base option pairs: {len(base_option_pairs)}")
    print(f"Total option pairs (with reversals): {len(option_pairs)}")

    # Generate all combinations: model × pair × temperature × capacity_context
    tasks_data = [
        (model, pair, temp, cap) 
        for model in MODELS 
        for pair in option_pairs 
        for temp in TEMPERATURE 
        for cap in CAPACITY_CONTEXTS
    ]
    print(f"Total tasks: {len(tasks_data)}")

    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    ssl_context = ssl.create_default_context(cafile=certifi.where())
    
    successful = 0
    failed = 0
    failed_tasks = []
    
    connector = aiohttp.TCPConnector(ssl=ssl_context)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [
            process_pair_async(session, model, pair, temp, cap, semaphore)
            for model, pair, temp, cap in tasks_data
        ]
        
        for coro in tqdm.as_completed(tasks, total=len(tasks), desc="Processing"):
            try:
                success, failed_info = await coro
                if success:
                    successful += 1
                else:
                    failed += 1
                    if failed_info:
                        failed_tasks.append(failed_info)
                    
            except Exception as e:
                print(f"[Error] Task failed with exception: {e}")
                failed += 1

    print(f"\nSimulation complete!")
    print(f"Successful: {successful}, Failed: {failed}")
    
    if failed_tasks:
        print(f"\n{'='*60}")
        print(f"Failed Tasks After 3 Retries ({len(failed_tasks)} total):")
        print(f"{'='*60}")
        for model, pair, temp, cap in failed_tasks:
            option_letters = [re.search(r"Option\s+([A-I])", o).group(1) for o in pair]
            print(f"  Model: {model}")
            print(f"  Pair: {option_letters[0]}-{option_letters[1]}")
            print(f"  Temperature: {temp}")
            print(f"  Capacity: {cap}%")
            print(f"  {'-'*58}")
    
    print(f"\nResults saved in {OUTPUT_DIR}/")
    for model in MODELS:
        print(f"  - results_{model.replace('/', '_').replace(':', '_')}.csv")


In [20]:

# === Main Entry Point ===
if __name__ == "__main__":
    try:
        get_ipython()
        import nest_asyncio
        nest_asyncio.apply()
    except (NameError, ImportError):
        pass
    
    asyncio.run(run_simulation_async())


Starting simulation with models: x-ai/grok-4-fast, google/gemini-2.5-flash, deepseek/deepseek-chat-v3.1, openai/gpt-5-nano
Temperature values: [0.1, 0.7]
Capacity contexts: [20, 50, 80]%
Max concurrent requests: 2
Base option pairs: 36
Total option pairs (with reversals): 72
Total tasks: 1728


Processing:   0%|          | 0/1728 [00:00<?, ?it/s]/var/folders/6n/bdd22kl9663flsjbqxx1cgdr0000gn/T/ipykernel_63465/3493630459.py:36: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"cap{capacity_context}_temp{temperature}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid.uuid4().hex[:8]}"
/var/folders/6n/bdd22kl9663flsjbqxx1cgdr0000gn/T/ipykernel_63465/3493630459.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
Processing:  44%|████▍     | 767/1728 [52:08<2:36:14,  9.75s/it]

[Error] Request failed: 


Processing:  62%|██████▏   | 1067/1728 [1:14:05<09:58,  1.10it/s]  

[Error] HTTP 429: {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"deepseek/deepseek-chat-v3.1 is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate yo


Processing:  62%|██████▎   | 1080/1728 [1:14:23<15:37,  1.45s/it]

... (9 outputs left out)

Processing:  97%|█████████▋| 1677/1728 [1:48:27<07:59,  9.41s/it]

[Error] HTTP 429: {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"deepseek/deepseek-chat-v3.1 is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate yo


Processing: 100%|██████████| 1728/1728 [1:52:23<00:00,  3.90s/it]


Simulation complete!
Successful: 1728, Failed: 0

Results saved in strategy_two_results/
  - results_x-ai_grok-4-fast.csv
  - results_google_gemini-2.5-flash.csv
  - results_deepseek_deepseek-chat-v3.1.csv
  - results_openai_gpt-5-nano.csv
